# stages 파일별 컬럼 설명표
이 노트북을 `C:\PHOTO_ANALYSIS\BASIC\Stages_Column_Dictionary.ipynb`에 놓고 위에서 아래로 실행합니다.

입력은 `output_decorrect_notebook\stages`이며, 결과는 새 `output_column_dictionary` 폴더에 저장합니다. **실제 데이터 본문을 로드하지 않고** Parquet schema/footer와 Excel 첫 행을 읽어 파일별 컬럼 표를 만듭니다. DB 연결과 별도 Python 모듈은 필요하지 않습니다.

설명은 기존 MRC/PSM/TROCS/WK/RK/CPE 노트북의 코드 기준입니다. 실제 존재하는 파일명·컬럼명·Parquet 자료형·파일 행 수를 기록하며, 미등록 컬럼은 따로 표시합니다. Excel 자료형은 헤더만으로 알 수 없으므로 미검사로 표시합니다.


## 00. 입력·출력 경로
Windows 기본 경로가 아래에 고정되어 있습니다. 필요하면 `BASE_DIR`만 수정하세요. 필요한 패키지는 pandas, pyarrow, openpyxl이며 기존 노트북 환경에서 사용한 패키지입니다.


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import pyarrow.parquet as pq
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from IPython.display import display, FileLink

BASE_DIR = Path(r"C:\PHOTO_ANALYSIS\BASIC")
STAGE_DIR = BASE_DIR / "output_decorrect_notebook" / "stages"
OUTPUT_DIR = BASE_DIR / "output_column_dictionary"
if not STAGE_DIR.is_dir():
    raise FileNotFoundError(f"입력 stages 폴더를 찾을 수 없습니다: {STAGE_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("입력:", STAGE_DIR)
print("결과:", OUTPUT_DIR)


## 01. 컬럼 의미와 모델 term 사전
파일별 설명의 기준을 여기서 직접 수정할 수 있습니다. `BASE_COLUMNS` 값은 설명 / 분류 / 단위 / 생성식 / 주의 순서입니다. 각 모델의 계수는 원본 matrix 식을 그대로 표시합니다.


In [ ]:
BASE_COLUMNS = {
    'LOT_ID': ['LOT 식별자. 같은 의미의 이름을 원본/가공 단계에 따라 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'lotid': ['LOT 식별자. 같은 의미의 이름을 원본/가공 단계에 따라 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'lot_id': ['LOT 식별자. 같은 의미의 이름을 원본/가공 단계에 따라 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'Wafer': ['LOT 내 wafer slot 번호.', '식별/공정', '정수 slot 번호', '원본 DB 기록값', ''],
    'slotid': ['LOT 내 wafer slot 번호.', '식별/공정', '정수 slot 번호', '원본 DB 기록값', ''],
    'slot_id': ['LOT 내 wafer slot 번호.', '식별/공정', '정수 slot 번호', '원본 DB 기록값', ''],
    'photo_transn_seq': ['노광 이력의 transaction 번호. ADI/OCO 공통 wafer 선정과 MRC/F1 연결에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', '공통 wafer 선정 키는 photo_transn_seq + slotid입니다.'],
    'lot_transn_seq': ['측정 LOT 이력의 transaction 번호. 원본 RAWDATA 조회·결합에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', 'ADI와 OCO의 번호가 다를 수 있습니다.'],
    'lot_transn_seq_oco': ['ADI/OCO paired 표에 보관한 OCO 측 lot transaction 번호.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'STEPSEQ': ['노광 공정 step 식별 코드.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'pstepseq': ['노광 공정 step 식별 코드.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'photo_step_seq': ['노광 공정 step 식별 코드.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'step_seq': ['노광 공정 step 식별 코드.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'M_STEPSEQ': ['측정 공정 step 식별 코드. ADI/OCO 구분 조건에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'mstepseq': ['측정 공정 step 식별 코드. ADI/OCO 구분 조건에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'metro_step_seq': ['측정 공정 step 식별 코드. ADI/OCO 구분 조건에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'mstepseq_oco': ['paired 표에 보관한 OCO 측 측정 공정 코드.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'P_EQPID': ['노광 장비 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'peqpid': ['노광 장비 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'eqp_id': ['노광 장비 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'Photo_PPID': ['노광 recipe/PPID 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'ppid': ['노광 recipe/PPID 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'ChuckID': ['LOTINFO에서 가져온 chuck 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', 'PSM 연결에 계산해서 사용하는 sub_eqp_id와 별도 컬럼입니다.'],
    'chuckid': ['LOTINFO에서 가져온 chuck 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', 'PSM 연결에 계산해서 사용하는 sub_eqp_id와 별도 컬럼입니다.'],
    'ReticleID': ['reticle 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'reticleid': ['reticle 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'reticle_id': ['reticle 식별자.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'Base_EQP1': ['F1 parameter 이력에서 연결한 base_eqp_id1 장비 정보.', '식별/공정', '해당 없음', '원본 DB 기록값', '구체적인 장비 업무 정의는 원 DB 필드 정의를 따릅니다.'],
    'base_eqp_id1': ['F1 parameter 이력에서 연결한 base_eqp_id1 장비 정보.', '식별/공정', '해당 없음', '원본 DB 기록값', '구체적인 장비 업무 정의는 원 DB 필드 정의를 따릅니다.'],
    'MMO_MRC_EQP': ['EXPO 이력의 MRC 참조 장비 ID.', '식별/공정', '해당 없음', '원본 DB 기록값', '15번에서는 이 값이 없는 실측 행을 제외합니다.'],
    'mmo_mrc_ref_eqp_id': ['EXPO 이력의 MRC 참조 장비 ID.', '식별/공정', '해당 없음', '원본 DB 기록값', '15번에서는 이 값이 없는 실측 행을 제외합니다.'],
    'psm_mmo_ref_eqp_id': ['LOTINFO의 PSM 참조 장비 정보. 원본 metadata를 보관합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'base_eqpid': ['per_shot_mrc_id 문자열의 마지막 경로 요소에서 첫 밑줄 앞 부분을 추출한 값.', '식별/공정', '해당 없음', "per_shot_mrc_id.split('/')[-1].split('_')[0]", 'F1에서 가져온 Base_EQP1과 별도 출처입니다.'],
    'per_shot_mrc_id': ['기적용 per-shot MRC 식별 문자열. paired 표의 장비/시각 문자열 추출에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'psm_update_time': ['per_shot_mrc_id에서 추출·가공한 PSM update 시각 문자열.', '시간', '문자열', "마지막 경로의 '_' 두 번째 부분 → 선두 20 제거 → 앞 10글자", 'DB timestamp를 직접 읽은 값이 아닙니다.'],
    'P_TIME': ['노광 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'photo_date': ['노광 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'M_TIME': ['LOTINFO의 측정 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'timestamp': ['LOTINFO의 측정 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'timestamp_oco': ['paired 표에 보관한 OCO 측 측정 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'impala_insert_time': ['DB 적재 시각. 최신 이력 선택 및 조회 기간 조건에 사용합니다.', '시간', 'DB datetime/문자열', '원본 DB 기록값', '공정/측정 시각과 다릅니다.'],
    'transn_occur_date': ['RAWDATA 종류별 batch를 구분하는 transaction 발생 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', 'LOT transaction/wafer/data_kind별 시각 차이가 60초 초과하면 새 batch로 구분하고 마지막 batch를 선택합니다.'],
    'event_tsdt': ['APC 이력 event 시각. 연결된 이력의 최신 순서를 정할 때 사용합니다.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'event_tsdt_eexy': ['EEXY/TROCS 그룹의 event 시각.', '시간', 'DB datetime/문자열', '원본 DB 기록값', ''],
    'tkin_tsdt': ['APC 이력의 tkin 시각. main/EEXY 최근접 연결 기준입니다.', '시간', 'DB datetime/문자열', '원본 DB 기록값', '키는 LOT/step/장비/sub_eqp이며 기본 허용 간격은 60초입니다.'],
    'db_user': ['DB 조회에 사용한 업무 계정/영역 구분값.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'rn': ['LOT/slot별 적재 시각 내림차순 row_number.', '선정/상태', '정수', 'row_number() over(partition by lotid,slotid order by impala_insert_time desc)', '조회 결과는 rn=1만 남깁니다.'],
    'seq_no': ['EXPO 이력의 sequence 번호. metadata로 보관합니다.', '식별/공정', '원본 DB 번호', '원본 DB 기록값', ''],
    'subname': ['PARAMDATA 종류. MRC 입력에서는 CMP 이력을 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'apc_group_name': ['APC 이력 그룹명. PHOTO_EEXY_GRP 여부로 main/EEXY 조회를 나눕니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'run_type': ['APC 이력의 run type metadata.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'sub_eqp_id': ['APC sub-equipment/chuck 식별자. 가공 실측 PSM 연결에서는 홀수 slot=EX1, 짝수 slot=EX2로 계산합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', '이력 조회 파일에서는 DB 원본값입니다.'],
    'apc_hist_index_no': ['APC 이력 번호. 조회 shot 파일은 EEXY/TROCS 번호, 가공 input/실측 자료는 main APC 번호입니다.', '식별/공정', '해당 없음', '원본 DB 기록값', '파일별 설명에서 번호 종류를 구분합니다.'],
    'apc_trocs_hist_index_no': ['연결된 EEXY/TROCS APC 이력 번호. 기적용 PSM/TROCS shot input 조회에 사용합니다.', '식별/공정', '해당 없음', '원본 DB 기록값', ''],
    'data_kind': ['원본 데이터 종류: RAWDATA=측정값, TESTDATA=mark 위치/관련 정보, PERSHOT=기적용 per-shot 보정값.', '원본 측정', '분류값', '원본 DB 기록값', ''],
    'TEST': ['측정 TEST point 번호.', '위치/그룹', '정수 ID', '원본 DB 기록값', 'TEST는 원본 test_point_no를 rename한 값입니다.'],
    'test_point_no': ['측정 TEST point 번호.', '위치/그룹', '정수 ID', '원본 DB 기록값', 'TEST는 원본 test_point_no를 rename한 값입니다.'],
    'GROUP': ['TEST 번호에서 정한 측정 그룹.', '위치/그룹', 'E1/E2/E3', 'pd.cut(TEST,[0,80,160,240]): 1~80=E1,81~160=E2,161~240=E3', 'OCO 공정 코드의 [2:5]가 077이면 E2, 그 외는 E1만 선택합니다.'],
    'DieX': ['wafer shot 배치의 X/Y index. 실제 chip 번호가 아니라 shot 중심을 계산하는 격자 index입니다.', '좌표', 'shot index', 'RAWDATA의 coordinate_x/y rename', ''],
    'DieY': ['wafer shot 배치의 X/Y index. 실제 chip 번호가 아니라 shot 중심을 계산하는 격자 index입니다.', '좌표', 'shot index', 'RAWDATA의 coordinate_x/y rename', ''],
    'Shot_X': ['TROCS 전체 mark 생성 시 사용하는 shot 배치 index.', '좌표', 'shot index', '원본 배치 후보 -10~10', ''],
    'Shot_Y': ['TROCS 전체 mark 생성 시 사용하는 shot 배치 index.', '좌표', 'shot index', '원본 배치 후보 -10~10', ''],
    'Test_ID': ['TROCS 전체 mark 좌표 목록에서 새로 부여한 1부터 시작하는 순번.', '위치/그룹', '생성 mark 순번', '정렬된 TEST별 첫 유효 mark 좌표 목록의 위치 + 1', '원래 TEST 번호와 동일하지 않습니다.'],
    'coordinate_x': ['원본 DB의 좌표 필드. RAWDATA/PERSHOT에서는 shot index로 사용합니다.', '좌표', '행 종류에 따른 DB 단위', '원본 DB 기록값', 'TESTDATA의 실제 local mark 좌표는 value_x/y에서 읽습니다. 가공 coordinate_X/Y와 대소문자를 구분하세요.'],
    'coordinate_y': ['원본 DB의 좌표 필드. RAWDATA/PERSHOT에서는 shot index로 사용합니다.', '좌표', '행 종류에 따른 DB 단위', '원본 DB 기록값', 'TESTDATA의 실제 local mark 좌표는 value_x/y에서 읽습니다. 가공 coordinate_X/Y와 대소문자를 구분하세요.'],
    'coordinate_X': ['shot 중심 기준 측정 mark의 local 좌표.', '좌표', 'µm', 'TESTDATA의 value_x/y rename', ''],
    'coordinate_Y': ['shot 중심 기준 측정 mark의 local 좌표.', '좌표', 'µm', 'TESTDATA의 value_x/y rename', ''],
    'Coordinate_X': ['TROCS 생성 mark의 shot-local 좌표.', '좌표', 'µm', '실측 TEST의 첫 유효 local 좌표를 wafer 각 shot에 반복 적용', ''],
    'Coordinate_Y': ['TROCS 생성 mark의 shot-local 좌표.', '좌표', 'µm', '실측 TEST의 첫 유효 local 좌표를 wafer 각 shot에 반복 적용', ''],
    'STEP_PITCH_X': ['shot 배치 pitch X/Y.', '좌표', 'µm', 'LOTINFO steppitch_x/y', 'DB 입력이 원본 코드에서 가정한 µm 단위인지 확인해야 합니다.'],
    'STEP_PITCH_Y': ['shot 배치 pitch X/Y.', '좌표', 'µm', 'LOTINFO steppitch_x/y', 'DB 입력이 원본 코드에서 가정한 µm 단위인지 확인해야 합니다.'],
    'steppitch_x': ['shot 배치 pitch X/Y.', '좌표', 'µm', 'LOTINFO steppitch_x/y', 'DB 입력이 원본 코드에서 가정한 µm 단위인지 확인해야 합니다.'],
    'steppitch_y': ['shot 배치 pitch X/Y.', '좌표', 'µm', 'LOTINFO steppitch_x/y', 'DB 입력이 원본 코드에서 가정한 µm 단위인지 확인해야 합니다.'],
    'MAP_SHIFT_X': ['shot 배치의 wafer 원점 기준 X/Y shift.', '좌표', 'µm', 'LOTINFO mapshift_x/y', ''],
    'MAP_SHIFT_Y': ['shot 배치의 wafer 원점 기준 X/Y shift.', '좌표', 'µm', 'LOTINFO mapshift_x/y', ''],
    'mapshift_x': ['shot 배치의 wafer 원점 기준 X/Y shift.', '좌표', 'µm', 'LOTINFO mapshift_x/y', ''],
    'mapshift_y': ['shot 배치의 wafer 원점 기준 X/Y shift.', '좌표', 'µm', 'LOTINFO mapshift_x/y', ''],
    'CHIP_X_NUM': ['shot 내부 X/Y 방향 chip 개수.', '좌표', '정수 개수', 'LOTINFO chip_x_qty/y_qty', 'CPE virtual grid를 생성할 때 사용합니다.'],
    'CHIP_Y_NUM': ['shot 내부 X/Y 방향 chip 개수.', '좌표', '정수 개수', 'LOTINFO chip_x_qty/y_qty', 'CPE virtual grid를 생성할 때 사용합니다.'],
    'chip_x_qty': ['shot 내부 X/Y 방향 chip 개수.', '좌표', '정수 개수', 'LOTINFO chip_x_qty/y_qty', 'CPE virtual grid를 생성할 때 사용합니다.'],
    'chip_y_qty': ['shot 내부 X/Y 방향 chip 개수.', '좌표', '정수 개수', 'LOTINFO chip_x_qty/y_qty', 'CPE virtual grid를 생성할 때 사용합니다.'],
    'Dmargin_X': ['LOTINFO의 X/Y margin 설정 metadata.', '설정', 'DB 원단위 확인 필요', '원본 DB 기록값', '현재 회귀·decorrect 계산에서는 직접 사용하지 않습니다. 물리적 상세 의미는 DB 정의를 따릅니다.'],
    'Dmargin_Y': ['LOTINFO의 X/Y margin 설정 metadata.', '설정', 'DB 원단위 확인 필요', '원본 DB 기록값', '현재 회귀·decorrect 계산에서는 직접 사용하지 않습니다. 물리적 상세 의미는 DB 정의를 따릅니다.'],
    'dmargin_x': ['LOTINFO의 X/Y margin 설정 metadata.', '설정', 'DB 원단위 확인 필요', '원본 DB 기록값', '현재 회귀·decorrect 계산에서는 직접 사용하지 않습니다. 물리적 상세 의미는 DB 정의를 따릅니다.'],
    'dmargin_y': ['LOTINFO의 X/Y margin 설정 metadata.', '설정', 'DB 원단위 확인 필요', '원본 DB 기록값', '현재 회귀·decorrect 계산에서는 직접 사용하지 않습니다. 물리적 상세 의미는 DB 정의를 따릅니다.'],
    'Outlier_Spec_Ratio': ['LOTINFO의 outlier/residual spec ratio metadata.', '설정', 'DB 설정 배율', '원본 DB 기록값', '이 노트북에서는 이 값으로 rework/양불 판정을 수행하지 않습니다.'],
    'outlr_resdl_spec_ratio': ['LOTINFO의 outlier/residual spec ratio metadata.', '설정', 'DB 설정 배율', '원본 DB 기록값', '이 노트북에서는 이 값으로 rework/양불 판정을 수행하지 않습니다.'],
    'fcp_x': ['wafer 원점 기준 shot 중심 X 좌표.', '좌표', 'µm', 'DieX * STEP_PITCH_X + MAP_SHIFT_X; TROCS는 Shot_X 사용', ''],
    'fcp_y': ['wafer 원점 기준 shot 중심 Y 좌표.', '좌표', 'µm', 'DieY * STEP_PITCH_Y + MAP_SHIFT_Y; TROCS는 Shot_Y 사용', ''],
    'fcp_x_round': ['연결 key 생성용 shot 중심 좌표를 소수점 2자리로 반올림한 값.', '좌표/연결', 'µm', 'round(fcp 또는 pos 좌표, 2)', ''],
    'fcp_y_round': ['연결 key 생성용 shot 중심 좌표를 소수점 2자리로 반올림한 값.', '좌표/연결', 'µm', 'round(fcp 또는 pos 좌표, 2)', ''],
    'wf_x': ['측정 point의 wafer 원점 기준 절대 X 좌표.', '좌표', 'µm', 'fcp_x + coordinate_X', ''],
    'wf_y': ['측정 point의 wafer 원점 기준 절대 Y 좌표.', '좌표', 'µm', 'fcp_y + coordinate_Y', ''],
    'Wafer_X': ['TROCS 생성 mark의 wafer 절대 X 좌표.', '좌표', 'µm', 'fcp_x + Coordinate_X', ''],
    'Wafer_Y': ['TROCS 생성 mark의 wafer 절대 Y 좌표.', '좌표', 'µm', 'fcp_y + Coordinate_Y', ''],
    'radius': ['실측 point와 wafer 원점 사이 거리.', '좌표', 'µm', 'sqrt(wf_x² + wf_y²)', ''],
    'Radius': ['TROCS 생성 mark와 wafer 원점 사이 거리.', '좌표', 'µm', 'sqrt(Wafer_X² + Wafer_Y²)', ''],
    'Inside_Radius': ['TROCS 생성 mark가 유효 반경 내부인지 여부.', '선정/상태', 'boolean', 'Radius < TROCS_WAFER_RADIUS (기본 147000 µm)', 'ALLMARKS에는 이 조건을 통과한 mark가 들어갑니다.'],
    'Max_Corner_Distance': ['shot의 네 모서리 중 wafer 원점과의 최대 거리.', '좌표', 'µm', 'max sqrt((fcp_x ± pitch_x/2)² + (fcp_y ± pitch_y/2)²)', ''],
    'Is_Edge_Shot': ['shot 모서리 하나 이상이 wafer 반경 밖인지 여부.', '선정/상태', 'boolean', 'Max_Corner_Distance > cfg.wafer_radius_um (기본 147000 µm)', 'CPE inner/partial 판정과 다른 기준입니다.'],
    'pos_x_valn': ['APC shot input에 기록된 shot 중심 좌표.', '좌표', 'QUERY=mm; 가공 INPUT=µm', '가공 input에서 DB 값 * 1000', '파일별 단위를 반드시 구분합니다.'],
    'pos_y_valn': ['APC shot input에 기록된 shot 중심 좌표.', '좌표', 'QUERY=mm; 가공 INPUT=µm', '가공 input에서 DB 값 * 1000', '파일별 단위를 반드시 구분합니다.'],
    'match_key': ['PSM 연결 key: LOT + EX1/EX2 + shot 중심 X/Y(소수점 2자리).', '연결 key', '문자열', 'LOT + sub_eqp_id + fmt2(fcp_x) + fmt2(fcp_y)', 'TROCS 연결 key와 다릅니다.'],
    'trocs_key': ['TROCS 연결 key: main APC 번호 + shot 중심 X/Y(소수점 2자리).', '연결 key', '문자열', 'main APC + fmt2(pos_x/fcp_x) + fmt2(pos_y/fcp_y)', '실측 point 연결에는 여기에 local 좌표 X/Y를 추가합니다.'],
    '_psm_lot_key': ['PSM 연결을 위해 LOT 문자열을 정리한 보조 컬럼.', '연결 key', '문자열', 'LOT_ID.astype(str).str.strip()', ''],
    '_psm_slot_key': ['PSM 연결을 위해 slot 번호를 nullable integer로 변환한 보조 컬럼.', '연결 key', '정수 slot 번호', 'integer(Wafer)', ''],
    'UNIQUE_ID3': ['공정·장비·시각·LOT·wafer metadata를 묶은 base 식별 문자열.', '연결 key', '문자열', 'STEPSEQ + P_EQPID + Photo_PPID + MMO_MRC_EQP + P_TIME + M_TIME + LOT_ID + Wafer', '실제 wafer를 구분할 때는 LOT_ID + Wafer도 확인하세요.'],
    'UNIQUE_ID': ['base 식별 문자열에 GROUP을 추가한 식별자.', '연결 key', '문자열', 'UNIQUE_ID3 + GROUP', '같은 wafer라도 GROUP이 다르면 UNIQUE_ID가 다릅니다.'],
    'UNIQUE_ID2': ['base 식별 문자열에 TEST·shot index·GROUP을 추가한 point 식별자.', '연결 key', '문자열', 'UNIQUE_ID3 + TEST + DieX + DieY + GROUP', ''],
    'UNIQUE_ID4': ['base 식별 문자열에 shot index·GROUP을 추가한 식별자.', '연결 key', '문자열', 'UNIQUE_ID3 + DieX + DieY + GROUP', ''],
    'value_x': ['원본 DB X/Y 값. data_kind에 따라 측정값/mark 좌표/기적용 per-shot 보정값으로 해석합니다.', '원본 측정', 'RAWDATA/PERSHOT=측정값 원단위; TESTDATA=µm', 'RAWDATA→X_reg/Y_reg; TESTDATA→coordinate_X/Y; PERSHOT→PSM_X/Y', ''],
    'value_y': ['원본 DB X/Y 값. data_kind에 따라 측정값/mark 좌표/기적용 per-shot 보정값으로 해석합니다.', '원본 측정', 'RAWDATA/PERSHOT=측정값 원단위; TESTDATA=µm', 'RAWDATA→X_reg/Y_reg; TESTDATA→coordinate_X/Y; PERSHOT→PSM_X/Y', ''],
    'mrc_x_valn': ['원본 데이터 행의 MRC 관련 X/Y 기록값.', '원본 측정', 'DB 원단위 확인 필요', '원본 DB 기록값', '가공 MRC_RX/RY는 TESTDATA 행에서 가져옵니다. CMP 계수로 계산한 mrc_fit과 구분하세요.'],
    'mrc_y_valn': ['원본 데이터 행의 MRC 관련 X/Y 기록값.', '원본 측정', 'DB 원단위 확인 필요', '원본 DB 기록값', '가공 MRC_RX/RY는 TESTDATA 행에서 가져옵니다. CMP 계수로 계산한 mrc_fit과 구분하세요.'],
    'MRC_RX': ['TESTDATA의 mrc_x_valn/mrc_y_valn을 보관한 값.', '원본 측정', 'DB 원단위 확인 필요', 'TESTDATA 필드 rename', '이 값은 현재 CMP MRC decorrect 계산에 직접 사용하지 않습니다.'],
    'MRC_RY': ['TESTDATA의 mrc_x_valn/mrc_y_valn을 보관한 값.', '원본 측정', 'DB 원단위 확인 필요', 'TESTDATA 필드 rename', '이 값은 현재 CMP MRC decorrect 계산에 직접 사용하지 않습니다.'],
    'X_reg': ['기적용 보정이 포함된 원본 X 방향 측정값.', '측정값', '측정값 원단위', 'RAWDATA value_x', ''],
    'PSM_X': ['ADI에서 실제 PERSHOT 행으로 가져온 X 방향 기적용 보정값.', '기적용 보정', '측정값 원단위', 'PERSHOT value_x', 'OCO에는 보통 NaN입니다. PSM 계수로 계산한 psm_fit과 구분하세요.'],
    'mrc_fit_x': ['CMP WK/RK 계수로 평가한 기적용 MRC X 방향 값(원본의 -1 부호 포함).', '기적용 보정', '측정값 원단위', '-1 * MRC_matrix_X @ CMP_coeff_X', ''],
    'X_reg_demrc': ['기적용 MRC를 되돌린 X 방향 측정값.', 'decorrect', '측정값 원단위', 'X_reg - mrc_fit_x', ''],
    'psm_fit_x': ['기적용 PSM 72계수로 평가한 X 방향 보정값.', '기적용 보정', '측정값 원단위', 'cpe_k_to_fit(local)_X @ applied_mrc_k_X', 'ADI raw 계산은 이 값 대신 실제 PERSHOT 값을 사용합니다.'],
    'raw_x': ['MRC와 PSM을 되돌린 X 방향 값. WK/RK 회귀 입력입니다.', 'decorrect', '측정값 원단위', 'ADI: X_reg_demrc - PSM_X; OCO: X_reg_demrc - psm_fit_x', 'TROCS를 추가 차감하지 않습니다. 최초 RAWDATA value와 구분하세요.'],
    'trocs_fit_all_x': ['wafer 전체 생성 mark에서 평가한 기적용 TROCS X 방향 값.', 'TROCS 연결', '측정값 원단위', 'cpe_k_to_fit(local)_X @ applied_rk1_to_72_X', ''],
    'trocs_fit_x': ['전체 mark TROCS 값 중 실측 point에 연결한 X 방향 값.', 'TROCS 연결', '측정값 원단위', 'APC+shot 중심+local 좌표로 trocs_fit_all_x 연결', '현재 파이프라인에서는 raw에서 빼지 않고 연결 값으로 보관합니다.'],
    'wk_fit_x': ['wafer WK/RK 동시 회귀에서 WK block이 설명하는 X 방향 point별 값.', 'WK/RK 제거', '측정값 원단위', 'MRC_matrix_X[:, :10] @ wafer_WK_coeff_X', ''],
    'rk_fit_x': ['wafer WK/RK 동시 회귀에서 RK block이 설명하는 X 방향 point별 값.', 'WK/RK 제거', '측정값 원단위', 'MRC_matrix_X[:, 10:] @ wafer_RK_coeff_X', 'shot별 CPE_fit과 구분하세요.'],
    'total_fit_x': ['WK와 RK point별 제거값의 합(X 방향).', 'WK/RK 제거', '측정값 원단위', 'wk_fit_x + rk_fit_x', ''],
    'raw_minus_total_fit_x': ['WK/RK 제거 후 X 방향 residual. shot CPE38 회귀 입력입니다.', 'WK/RK residual', '측정값 원단위', 'raw_x - total_fit_x', ''],
    'CPE_fit_x': ['해당 shot의 CPE38 계수로 설명하는 X 방향 point별 제거값.', 'CPE 제거', '측정값 원단위', 'CPE38_matrix_X @ shot_CPE_coeff_X', ''],
    'real_residual_x': ['WK/RK 및 shot CPE38를 제거한 최종 X 방향 residual.', '최종 residual', '측정값 원단위', 'raw_minus_total_fit_x - CPE_fit_x', ''],
    'CPE_gopt_x': ['해당 shot X 방향 회귀의 최종 G-opt 지표.', 'CPE 진단', '현재 정의의 수치 지표', 'Type1 기본: sqrt(max(sum(H², axis=1)))', 'box 제약 후에도 같은 lambda의 선형 ridge hat matrix로 계산합니다.'],
    'gopt_x': ['해당 shot X 방향 회귀의 최종 G-opt 지표.', 'CPE 진단', '현재 정의의 수치 지표', 'Type1 기본: sqrt(max(sum(H², axis=1)))', 'box 제약 후에도 같은 lambda의 선형 ridge hat matrix로 계산합니다.'],
    'CPE_lambda_max_x': ['해당 shot X 방향 coefficient별 ridge lambda 벡터의 최대값.', 'CPE 진단', 'penalty 스케일', 'max(lambda_vector)', '각 term의 lambda_RK*_X/Y와 구분하세요.'],
    'lambda_max_x': ['해당 shot X 방향 coefficient별 ridge lambda 벡터의 최대값.', 'CPE 진단', 'penalty 스케일', 'max(lambda_vector)', '각 term의 lambda_RK*_X/Y와 구분하세요.'],
    'CPE_method_x': ['해당 shot X 방향에 사용한 CPE 계산 방법.', 'CPE 진단', '분류값', 'ols / ridge_gopt / ridge_gopt_box_cd', 'method 이름은 계산 분기를 나타내며 spec 통과를 보장하지 않습니다.'],
    'method_x': ['해당 shot X 방향에 사용한 CPE 계산 방법.', 'CPE 진단', '분류값', 'ols / ridge_gopt / ridge_gopt_box_cd', 'method 이름은 계산 분기를 나타내며 spec 통과를 보장하지 않습니다.'],
    'gopt_iter_x': ['X 방향 ridge 확대와 이분 탐색에서 기록한 G-opt 반복 수.', 'CPE 진단', '정수 횟수', '원본 DB 기록값', ''],
    'span_iter_x': ['X 방향 box coordinate descent 반복 수. 실행하지 않으면 0.', 'CPE 진단', '정수 횟수', '원본 DB 기록값', '횟수 자체는 수렴 보증이 아닙니다.'],
    'span_violation_count_x': ['X 방향에서 최종 contribution span이 현재 limit/tolerance를 위반한 term 수.', 'CPE 진단', '정수 term 수', '원본 DB 기록값', ''],
    'gopt_within_spec_x': ['X 방향 G-opt가 현재 spec 안에 있는지 여부.', 'CPE 진단', 'boolean', 'isfinite(gopt_x) and 0 < gopt_x < G_OPT_SPEC', ''],
    'span_within_limits_x': ['X 방향에 최종 span violation이 없는지 여부.', 'CPE 진단', 'boolean', 'span_violation_count_x == 0', ''],
    'rank_x': ['wafer WK/RK X 방향 design matrix의 수치 rank.', 'WK/RK 진단', '정수 rank', 'np.linalg.matrix_rank(matrix)', '19보다 작으면 계수 해가 유일하지 않을 수 있습니다.'],
    'rms_raw_x': ['wafer WK/RK 회귀에 사용한 raw X 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_x²))', ''],
    'rms_residual_x': ['wafer WK/RK 제거 후 X 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_minus_total_fit_x²))', ''],
    'rms_before_x': ['shot CPE 제거 전 X 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_minus_total_fit_x²))', ''],
    'rms_after_x': ['shot CPE 제거 후 X 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(real_residual_x²))', ''],
    'Y_reg': ['기적용 보정이 포함된 원본 Y 방향 측정값.', '측정값', '측정값 원단위', 'RAWDATA value_y', ''],
    'PSM_Y': ['ADI에서 실제 PERSHOT 행으로 가져온 Y 방향 기적용 보정값.', '기적용 보정', '측정값 원단위', 'PERSHOT value_y', 'OCO에는 보통 NaN입니다. PSM 계수로 계산한 psm_fit과 구분하세요.'],
    'mrc_fit_y': ['CMP WK/RK 계수로 평가한 기적용 MRC Y 방향 값(원본의 -1 부호 포함).', '기적용 보정', '측정값 원단위', '-1 * MRC_matrix_Y @ CMP_coeff_Y', ''],
    'Y_reg_demrc': ['기적용 MRC를 되돌린 Y 방향 측정값.', 'decorrect', '측정값 원단위', 'Y_reg - mrc_fit_y', ''],
    'psm_fit_y': ['기적용 PSM 72계수로 평가한 Y 방향 보정값.', '기적용 보정', '측정값 원단위', 'cpe_k_to_fit(local)_Y @ applied_mrc_k_Y', 'ADI raw 계산은 이 값 대신 실제 PERSHOT 값을 사용합니다.'],
    'raw_y': ['MRC와 PSM을 되돌린 Y 방향 값. WK/RK 회귀 입력입니다.', 'decorrect', '측정값 원단위', 'ADI: Y_reg_demrc - PSM_Y; OCO: Y_reg_demrc - psm_fit_y', 'TROCS를 추가 차감하지 않습니다. 최초 RAWDATA value와 구분하세요.'],
    'trocs_fit_all_y': ['wafer 전체 생성 mark에서 평가한 기적용 TROCS Y 방향 값.', 'TROCS 연결', '측정값 원단위', 'cpe_k_to_fit(local)_Y @ applied_rk1_to_72_Y', ''],
    'trocs_fit_y': ['전체 mark TROCS 값 중 실측 point에 연결한 Y 방향 값.', 'TROCS 연결', '측정값 원단위', 'APC+shot 중심+local 좌표로 trocs_fit_all_y 연결', '현재 파이프라인에서는 raw에서 빼지 않고 연결 값으로 보관합니다.'],
    'wk_fit_y': ['wafer WK/RK 동시 회귀에서 WK block이 설명하는 Y 방향 point별 값.', 'WK/RK 제거', '측정값 원단위', 'MRC_matrix_Y[:, :10] @ wafer_WK_coeff_Y', ''],
    'rk_fit_y': ['wafer WK/RK 동시 회귀에서 RK block이 설명하는 Y 방향 point별 값.', 'WK/RK 제거', '측정값 원단위', 'MRC_matrix_Y[:, 10:] @ wafer_RK_coeff_Y', 'shot별 CPE_fit과 구분하세요.'],
    'total_fit_y': ['WK와 RK point별 제거값의 합(Y 방향).', 'WK/RK 제거', '측정값 원단위', 'wk_fit_y + rk_fit_y', ''],
    'raw_minus_total_fit_y': ['WK/RK 제거 후 Y 방향 residual. shot CPE38 회귀 입력입니다.', 'WK/RK residual', '측정값 원단위', 'raw_y - total_fit_y', ''],
    'CPE_fit_y': ['해당 shot의 CPE38 계수로 설명하는 Y 방향 point별 제거값.', 'CPE 제거', '측정값 원단위', 'CPE38_matrix_Y @ shot_CPE_coeff_Y', ''],
    'real_residual_y': ['WK/RK 및 shot CPE38를 제거한 최종 Y 방향 residual.', '최종 residual', '측정값 원단위', 'raw_minus_total_fit_y - CPE_fit_y', ''],
    'CPE_gopt_y': ['해당 shot Y 방향 회귀의 최종 G-opt 지표.', 'CPE 진단', '현재 정의의 수치 지표', 'Type1 기본: sqrt(max(sum(H², axis=1)))', 'box 제약 후에도 같은 lambda의 선형 ridge hat matrix로 계산합니다.'],
    'gopt_y': ['해당 shot Y 방향 회귀의 최종 G-opt 지표.', 'CPE 진단', '현재 정의의 수치 지표', 'Type1 기본: sqrt(max(sum(H², axis=1)))', 'box 제약 후에도 같은 lambda의 선형 ridge hat matrix로 계산합니다.'],
    'CPE_lambda_max_y': ['해당 shot Y 방향 coefficient별 ridge lambda 벡터의 최대값.', 'CPE 진단', 'penalty 스케일', 'max(lambda_vector)', '각 term의 lambda_RK*_X/Y와 구분하세요.'],
    'lambda_max_y': ['해당 shot Y 방향 coefficient별 ridge lambda 벡터의 최대값.', 'CPE 진단', 'penalty 스케일', 'max(lambda_vector)', '각 term의 lambda_RK*_X/Y와 구분하세요.'],
    'CPE_method_y': ['해당 shot Y 방향에 사용한 CPE 계산 방법.', 'CPE 진단', '분류값', 'ols / ridge_gopt / ridge_gopt_box_cd', 'method 이름은 계산 분기를 나타내며 spec 통과를 보장하지 않습니다.'],
    'method_y': ['해당 shot Y 방향에 사용한 CPE 계산 방법.', 'CPE 진단', '분류값', 'ols / ridge_gopt / ridge_gopt_box_cd', 'method 이름은 계산 분기를 나타내며 spec 통과를 보장하지 않습니다.'],
    'gopt_iter_y': ['Y 방향 ridge 확대와 이분 탐색에서 기록한 G-opt 반복 수.', 'CPE 진단', '정수 횟수', '원본 DB 기록값', ''],
    'span_iter_y': ['Y 방향 box coordinate descent 반복 수. 실행하지 않으면 0.', 'CPE 진단', '정수 횟수', '원본 DB 기록값', '횟수 자체는 수렴 보증이 아닙니다.'],
    'span_violation_count_y': ['Y 방향에서 최종 contribution span이 현재 limit/tolerance를 위반한 term 수.', 'CPE 진단', '정수 term 수', '원본 DB 기록값', ''],
    'gopt_within_spec_y': ['Y 방향 G-opt가 현재 spec 안에 있는지 여부.', 'CPE 진단', 'boolean', 'isfinite(gopt_y) and 0 < gopt_y < G_OPT_SPEC', ''],
    'span_within_limits_y': ['Y 방향에 최종 span violation이 없는지 여부.', 'CPE 진단', 'boolean', 'span_violation_count_y == 0', ''],
    'rank_y': ['wafer WK/RK Y 방향 design matrix의 수치 rank.', 'WK/RK 진단', '정수 rank', 'np.linalg.matrix_rank(matrix)', '19보다 작으면 계수 해가 유일하지 않을 수 있습니다.'],
    'rms_raw_y': ['wafer WK/RK 회귀에 사용한 raw Y 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_y²))', ''],
    'rms_residual_y': ['wafer WK/RK 제거 후 Y 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_minus_total_fit_y²))', ''],
    'rms_before_y': ['shot CPE 제거 전 Y 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(raw_minus_total_fit_y²))', ''],
    'rms_after_y': ['shot CPE 제거 후 Y 방향 RMS.', '통계', '측정값 원단위', 'sqrt(mean(real_residual_y²))', ''],
    'mrc_status': ['기적용 CMP MRC 계수 연결/계산 상태.', '선정/상태', '분류값', 'matched / missing_mrc / invalid_mrc_k', 'matched라도 좌표 결측으로 계산값이 NaN일 수 있으므로 decorrect_valid도 확인하세요.'],
    'psm_fit_status': ['기적용 PSM 계수 연결/계산 상태.', '선정/상태', '분류값', 'matched / missing_psm / invalid_psm_k / invalid_psm_coordinates', ''],
    'psm_source': ['raw 계산에서 사용한 PSM 값의 출처.', '선정/상태', '분류값', 'ADI=PERSHOT; OCO=APC_PSM_FIT', ''],
    'decorrect_valid': ['raw_x와 raw_y가 모두 유한한 값인지 여부.', '선정/상태', 'boolean', 'isfinite([raw_x,raw_y]).all()', 'wafer 양불/rework 판정값이 아닙니다.'],
    'trocs_status': ['기적용 TROCS 계수/배치/연결 상태.', '선정/상태', '분류값', 'matched / partial_coefficients_zero_filled / all_coefficients_nan / missing_coefficients / invalid_layout / missing_apc / no_valid_marks / unmatched', '부분 NaN 계수는 원본처럼 0으로 계산합니다. ALLMARKS에는 보정값 없는 shot을 제외하고 DIAGNOSTICS에는 남깁니다.'],
    'mode': ['ADI/OCO 구분.', '식별/공정', 'ADI/OCO', '원본 DB 기록값', ''],
    'axis': ['raw 분해식 검증 방향.', '검증', 'x/y', '원본 DB 기록값', ''],
    'rows': ['15번 wafer 요약의 해당 wafer 전체 실측 행 수.', '집계', '정수 행 수', 'group size', ''],
    'mrc_matched': ['15번 wafer 요약에서 mrc_status=matched인 행 수.', '집계', '정수 행 수', '(mrc_status == matched).sum()', ''],
    'psm_fit_matched': ['15번 wafer 요약에서 psm_fit_status=matched인 행 수.', '집계', '정수 행 수', '(psm_fit_status == matched).sum()', ''],
    'valid_raw': ['15번 wafer 요약에서 decorrect_valid=True인 행 수.', '집계', '정수 행 수', 'decorrect_valid.sum()', ''],
    'invalid_raw': ['15번 wafer 요약에서 decorrect_valid=False인 행 수.', '집계', '정수 행 수', 'rows - valid_raw', ''],
    'n_total': ['해당 wafer/shot 회귀 그룹의 전체 입력 행 수.', '집계', '정수 행 수', 'len(group)', ''],
    'n_valid': ['회귀에 필요한 X/Y 값과 좌표의 결측 조건을 모두 통과한 행 수.', '집계', '정수 행 수', 'WK/RK: raw_x/y+fcp_x/y+local X/Y; CPE: WK/RK residual X/Y+local X/Y', '컬럼별 통계의 *_n_valid와 다른 기준입니다. 원본 회귀 mask는 notna 기준입니다.'],
    'n_points': ['wafer/shot 통계에 포함된 전체 point 행 수.', '집계', '정수 행 수', 'group size', '측정값 NaN 행도 전체 행 수에는 포함됩니다.'],
    'n_generated': ['TROCS 해당 shot에 생성한 유효 반경 내부 mark 수.', '집계', '정수 mark 수', '원본 DB 기록값', ''],
    'n_fitted': ['생성 TROCS mark 중 X/Y 보정값 하나 이상이 결측이 아닌 행 수.', '집계', '정수 mark 수', '원본 DB 기록값', ''],
    'n_parameters_per_axis': ['wafer WK/RK 각 방향 design matrix의 parameter 수.', 'WK/RK 진단', '정수 개수', '19 (=WK 10 + RK 9)', ''],
    'rank_deficient': ['WK/RK matrix의 rank가 X/Y 하나라도 parameter 수보다 작은지 여부.', 'WK/RK 진단', 'boolean', 'rank_x < 19 or rank_y < 19', ''],
    'n_virtual_full': ['wafer 반경 clipping 전 CPE virtual point 수.', 'CPE 진단', '정수 point 수', '원본 DB 기록값', ''],
    'n_virtual': ['CPE 평가에 실제 사용한 virtual point 수.', 'CPE 진단', '정수 point 수', 'chip grid → R<=144000 µm clipping; metadata 없으면 measured-range grid', 'clipping 후 0개이면 원본처럼 full grid로 fallback합니다.'],
    'virtual_source': ['CPE 평가 grid가 만들어진 방식.', 'CPE 진단', '분류값', 'chip_corner_center_wafer_clipped / chip_corner_center_no_clip_points_fallback / measured_range_grid_fallback / measured_range_grid', ''],
    'inner_or_partial': ['현재 CPE 코드의 inner/partial 구분.', 'CPE 진단', 'inner/partial', 'n_valid >= n_virtual이면 inner, 아니면 partial', '기하학적으로 내부/edge shot인지 직접 판정한 값이 아닙니다.'],
    'CPE_inner_or_partial': ['현재 CPE 코드의 inner/partial 구분.', 'CPE 진단', 'inner/partial', 'n_valid >= n_virtual이면 inner, 아니면 partial', '기하학적으로 내부/edge shot인지 직접 판정한 값이 아닙니다.'],
    'metro_ratio': ['유효 측정점 수와 평가 virtual point 수의 비.', 'CPE 진단', '비율', 'n_valid / max(n_virtual,1)', '1을 초과할 수 있습니다.'],
    'few_points_flag': ['CPE shot의 유효 측정점 수가 guide 값 미만인지 여부.', 'CPE 진단', 'boolean', 'n_valid < MIN_POINTS_GUIDE (기본 10)', '1개 이상이면 원본처럼 회귀합니다. 0개만 no_valid_points로 생략합니다.'],
    'base_lambda_scale': ['CPE 측정점 수에 따른 초기 ridge 탐색 배율.', 'CPE 진단', '배율', 'n>=10:1;7~9:1e2;5~6:1e4;3~4:1e6;1~2:1e8', '최종 lambda 자체가 아니라 탐색 시작 배율입니다.'],
    'status': ['회귀 계산 완료/생략 상태.', '선정/상태', '분류값', 'WK/RK: ok / skip_too_few_points; CPE: ok / no_valid_points', 'ok는 계산 완료 의미입니다. 수치 spec 통과나 wafer 양불을 의미하지 않습니다.'],
    'cpe_quality_ok': ['CPE 계산 완료 및 X/Y G-opt/span 조건을 모두 충족했는지 여부.', 'CPE 진단', 'boolean', 'status=ok and G-opt X/Y within spec and span X/Y violation=0', '연구용 회귀 진단입니다. rework/양불 기준이 아닙니다.'],
    'quality_status': ['CPE 회귀 진단을 모은 상태 문자열.', 'CPE 진단', '분류값', 'within_current_specs / review_gopt_or_span / no_valid_points', ''],
    'n_checked': ['raw 분해식의 다섯 값이 모두 결측이 아니어서 검증한 행 수.', '검증', '정수 행 수', '원본 DB 기록값', ''],
    'n_unchecked': ['필요한 제거값/잔차가 결측이라 분해식을 확인하지 않은 행 수.', '검증', '정수 행 수', 'n_total - n_checked', ''],
    'max_abs_error': ['검증한 raw와 WK+RK+CPE+최종 residual 합의 최대 절대 차이.', '검증', '측정값 원단위', 'max(abs(raw - (WK_fit+RK_fit+CPE_fit+real_residual)))', ''],
    'passed': ['해당 ADI/OCO 방향의 검증 대상 행이 모두 분해식 tolerance를 통과했는지 여부.', '검증', 'boolean', 'np.isclose(...,rtol=1e-10,atol=1e-12).all()', '검증 대상 행이 0개이면 False입니다.'],
    'check_status': ['분해식 검증 실행 여부.', '검증', '분류값', 'checked / no_valid_points', ''],
}

WAFER_TERMS = {
    'WK1_X': '1',
    'WK3_X': 'x / 1000000.0',
    'WK5_X': '-y / 1000000.0',
    'WK7_X': 'x ** 2 / 1000000000000.0',
    'WK9_X': 'x * y / 1000000000000.0',
    'WK11_X': 'y ** 2 / 1000000000000.0',
    'WK13_X': 'x ** 3 / 1000000000000000.0',
    'WK15_X': 'x ** 2 * y / 1000000000000000.0',
    'WK17_X': 'x * y ** 2 / 1000000000000000.0',
    'WK19_X': 'y ** 3 / 1000000000000000.0',
    'RK3_X': 'rx / 1000000.0',
    'RK5_X': '-ry / 1000000.0',
    'RK7_X': 'rx ** 2 / 1000000000.0',
    'RK9_X': 'rx * ry / 1000000000.0',
    'RK11_X': 'ry ** 2 / 1000000000.0',
    'RK13_X': 'rx ** 3 / 1000000000000.0',
    'RK15_X': 'rx ** 2 * ry / 1000000000000.0',
    'RK17_X': 'rx * ry ** 2 / 1000000000000.0',
    'RK19_X': 'ry ** 3 / 1000000000000.0',
    'WK2_Y': '1',
    'WK4_Y': 'y / 1000000.0',
    'WK6_Y': 'x / 1000000.0',
    'WK8_Y': 'y ** 2 / 1000000000000.0',
    'WK10_Y': 'y * x / 1000000000000.0',
    'WK12_Y': 'x ** 2 / 1000000000000.0',
    'WK14_Y': 'y ** 3 / 1000000000000000.0',
    'WK16_Y': 'y ** 2 * x / 1000000000000000.0',
    'WK18_Y': 'y * x ** 2 / 1000000000000000.0',
    'WK20_Y': 'x ** 3 / 1000000000000000.0',
    'RK4_Y': 'ry / 1000000.0',
    'RK6_Y': 'rx / 1000000.0',
    'RK8_Y': 'ry ** 2 / 1000000000.0',
    'RK10_Y': 'ry * rx / 1000000000.0',
    'RK12_Y': 'rx ** 2 / 1000000000.0',
    'RK14_Y': 'ry ** 3 / 1000000000000.0',
    'RK16_Y': 'ry ** 2 * rx / 1000000000000.0',
    'RK18_Y': 'ry * rx ** 2 / 1000000000000.0',
    'RK20_Y': 'rx ** 3 / 1000000000000.0',
}

APPLIED_TERMS = {
    'RK1_X': '1',
    'RK3_X': 'rx / 1000000.0',
    'RK5_X': '-ry / 1000000.0',
    'RK7_X': 'rx ** 2 / 1000000000.0',
    'RK9_X': 'rx * ry / 1000000000.0',
    'RK11_X': 'ry ** 2 / 1000000000.0',
    'RK13_X': 'rx ** 3 / 1000000000000.0',
    'RK15_X': 'rx ** 2 * ry / 1000000000000.0',
    'RK17_X': 'rx * ry ** 2 / 1000000000000.0',
    'RK19_X': 'ry ** 3 / 1000000000000.0',
    'RK21_X': 'rx ** 4 / 1e+19',
    'RK23_X': 'rx ** 3 * ry / 1e+19',
    'RK25_X': 'rx ** 2 * ry ** 2 / 1e+19',
    'RK27_X': 'rx * ry ** 3 / 1e+19',
    'RK29_X': 'ry ** 4 / 1e+19',
    'RK31_X': 'rx ** 5 / 1e+23',
    'RK33_X': 'rx ** 4 * ry / 1e+23',
    'RK35_X': 'rx ** 3 * ry ** 2 / 1e+23',
    'RK37_X': 'rx ** 2 * ry ** 3 / 1e+23',
    'RK39_X': 'rx * ry ** 4 / 1e+23',
    'RK41_X': 'ry ** 5 / 1e+23',
    'RK43_X': 'rx ** 6 / 1e+27',
    'RK45_X': 'rx ** 5 * ry / 1e+27',
    'RK47_X': 'rx ** 4 * ry ** 2 / 1e+27',
    'RK49_X': 'rx ** 3 * ry ** 3 / 1e+27',
    'RK51_X': 'rx ** 2 * ry ** 4 / 1e+27',
    'RK53_X': 'rx * ry ** 5 / 1e+27',
    'RK55_X': 'ry ** 6 / 1e+27',
    'RK57_X': 'rx ** 7 / 1e+31',
    'RK59_X': 'rx ** 6 * ry / 1e+31',
    'RK61_X': 'rx ** 5 * ry ** 2 / 1e+31',
    'RK63_X': 'rx ** 4 * ry ** 3 / 1e+31',
    'RK65_X': 'rx ** 3 * ry ** 4 / 1e+31',
    'RK67_X': 'rx ** 2 * ry ** 5 / 1e+31',
    'RK69_X': 'rx * ry ** 6 / 1e+31',
    'RK71_X': 'ry ** 7 / 1e+31',
    'RK2_Y': '1',
    'RK4_Y': 'ry / 1000000.0',
    'RK6_Y': 'rx / 1000000.0',
    'RK8_Y': 'ry ** 2 / 1000000000.0',
    'RK10_Y': 'ry * rx / 1000000000.0',
    'RK12_Y': 'rx ** 2 / 1000000000.0',
    'RK14_Y': 'ry ** 3 / 1000000000000.0',
    'RK16_Y': 'ry ** 2 * rx / 1000000000000.0',
    'RK18_Y': 'ry * rx ** 2 / 1000000000000.0',
    'RK20_Y': 'rx ** 3 / 1000000000000.0',
    'RK22_Y': 'ry ** 4 / 1e+19',
    'RK24_Y': 'ry ** 3 * rx / 1e+19',
    'RK26_Y': 'ry ** 2 * rx ** 2 / 1e+19',
    'RK28_Y': 'ry * rx ** 3 / 1e+19',
    'RK30_Y': 'rx ** 4 / 1e+19',
    'RK32_Y': 'ry ** 5 / 1e+23',
    'RK34_Y': 'ry ** 4 * rx / 1e+23',
    'RK36_Y': 'ry ** 3 * rx ** 2 / 1e+23',
    'RK38_Y': 'ry ** 2 * rx ** 3 / 1e+23',
    'RK40_Y': 'ry * rx ** 4 / 1e+23',
    'RK42_Y': 'rx ** 5 / 1e+23',
    'RK44_Y': 'ry ** 6 / 1e+27',
    'RK46_Y': 'ry ** 5 * rx / 1e+27',
    'RK48_Y': 'ry ** 4 * rx ** 2 / 1e+27',
    'RK50_Y': 'ry ** 3 * rx ** 3 / 1e+27',
    'RK52_Y': 'ry ** 2 * rx ** 4 / 1e+27',
    'RK54_Y': 'ry * rx ** 5 / 1e+27',
    'RK56_Y': 'rx ** 6 / 1e+27',
    'RK58_Y': 'ry ** 7 / 1e+31',
    'RK60_Y': 'ry ** 6 * rx / 1e+31',
    'RK62_Y': 'ry ** 5 * rx ** 2 / 1e+31',
    'RK64_Y': 'ry ** 4 * rx ** 3 / 1e+31',
    'RK66_Y': 'ry ** 3 * rx ** 4 / 1e+31',
    'RK68_Y': 'ry ** 2 * rx ** 5 / 1e+31',
    'RK70_Y': 'ry * rx ** 6 / 1e+31',
    'RK72_Y': 'rx ** 7 / 1e+31',
}

CPE_TERMS = {
    'RK1_X': '1',
    'RK3_X': 'rx / 1000000.0',
    'RK5_X': '-ry / 1000000.0',
    'RK7_X': 'rx ** 2 / 1000000000.0',
    'RK9_X': 'rx * ry / 1000000000.0',
    'RK11_X': 'ry ** 2 / 1000000000.0',
    'RK13_X': 'rx ** 3 / 1000000000000.0',
    'RK15_X': 'rx ** 2 * ry / 1000000000000.0',
    'RK17_X': 'rx * ry ** 2 / 1000000000000.0',
    'RK19_X': 'ry ** 3 / 1000000000000.0',
    'RK23_X': 'rx ** 3 * ry / 1e+19',
    'RK25_X': 'rx ** 2 * ry ** 2 / 1e+19',
    'RK27_X': 'rx * ry ** 3 / 1e+19',
    'RK29_X': 'ry ** 4 / 1e+19',
    'RK35_X': 'rx ** 3 * ry ** 2 / 1e+23',
    'RK37_X': 'rx ** 2 * ry ** 3 / 1e+23',
    'RK39_X': 'rx * ry ** 4 / 1e+23',
    'RK41_X': 'ry ** 5 / 1e+23',
    'RK49_X': 'rx ** 3 * ry ** 3 / 1e+27',
    'RK51_X': 'rx ** 2 * ry ** 4 / 1e+27',
    'RK65_X': 'rx ** 3 * ry ** 4 / 1e+31',
    'RK2_Y': '1',
    'RK4_Y': 'ry / 1000000.0',
    'RK6_Y': 'rx / 1000000.0',
    'RK8_Y': 'ry ** 2 / 1000000000.0',
    'RK10_Y': 'ry * rx / 1000000000.0',
    'RK12_Y': 'rx ** 2 / 1000000000.0',
    'RK14_Y': 'ry ** 3 / 1000000000000.0',
    'RK16_Y': 'ry ** 2 * rx / 1000000000000.0',
    'RK18_Y': 'ry * rx ** 2 / 1000000000000.0',
    'RK22_Y': 'ry ** 4 / 1e+19',
    'RK24_Y': 'ry ** 3 * rx / 1e+19',
    'RK26_Y': 'ry ** 2 * rx ** 2 / 1e+19',
    'RK32_Y': 'ry ** 5 / 1e+23',
    'RK34_Y': 'ry ** 4 * rx / 1e+23',
    'RK36_Y': 'ry ** 3 * rx ** 2 / 1e+23',
    'RK46_Y': 'ry ** 5 * rx / 1e+27',
    'RK48_Y': 'ry ** 4 * rx ** 2 / 1e+27',
}


## 02. 파일별·컬럼별 설명 함수
같은 RK 계수명을 wafer 회귀와 shot CPE 파일에 따라 구분합니다. QUERY와 INPUT의 pos 단위 및 APC 번호 종류도 구분합니다. 통계 suffix는 원래 변수에 대한 평균·표준편차·RMS·최대 절대값·유효값 개수를 설명합니다.


In [ ]:
STAT_NAMES = {"n_valid": "결측이 아닌 유효값 개수", "mean": "평균", "std": "표본 표준편차", "rms": "RMS", "max_abs": "최대 절대값"}
STAT_FORMULAS = {"n_valid": "count(value), NaN 제외", "mean": "mean(value), NaN 제외", "std": "std(value, ddof=1), NaN 제외", "rms": "sqrt(mean(value²)), NaN 제외", "max_abs": "max(abs(value)), NaN 제외"}
SOURCE_SNAPSHOTS = {"02_LOTINFO_OCO_QUERY": "lotinfo_oco", "03_LOTINFO_ADI_QUERY": "lotinfo_adi", "05_RAWDATA_OCO_QUERY": "rawdata_oco", "05_RAWDATA_ADI_QUERY": "rawdata_adi", "07_F1_QUERY": "f1", "11_APC_MAIN_QUERY": "apc_main", "11_APC_EEXY_QUERY": "apc_eexy", "12_PSM_SHOT_QUERY": "psm_shot"}
SCHEMA_SHEET_COLUMNS = ["파일명", "단계명", "컬럼순서", "컬럼명", "분류", "설명", "단위·스케일", "출처·생성식", "해석·주의", "설명등록여부", "실제자료형"]

def normalize_stage(filename):
    return re.sub(r"_sample_\d+$", "", Path(filename).stem)

def describe_column(column, stage=""):
    stage = normalize_stage(stage)
    for stat in STAT_NAMES:
        suffix = "_" + stat
        if column.endswith(suffix) and column[:-len(suffix)] in BASE_COLUMNS:
            original = column[:-len(suffix)]
            desc, _, unit, _, note = BASE_COLUMNS[original]
            note = "집계 기준: WAFER_DATA=LOT_ID+Wafer; SHOT_DATA=LOT_ID+Wafer+fcp_x/y. " + note
            if stat == "std":
                note += " 표본 표준편차이므로 유효값 1개 이하면 NaN입니다."
            return [f"{original}({desc})의 {STAT_NAMES[stat]}.", "통계", "정수 개수" if stat == "n_valid" else unit, STAT_FORMULAS[stat], note, True]
    coeff = re.fullmatch(r"(WK|RK)(\d+)_([XY])", column)
    if coeff:
        model = "CPE" if "CPE_SHOT_COEFFICIENTS" in stage else "WAFER" if "WAFER_COEFFICIENTS" in stage else None
        terms = CPE_TERMS if model == "CPE" else WAFER_TERMS if model == "WAFER" else {}
        if column not in terms:
            return ["WK/RK 계수 이름입니다. 현재 코드 기준 파일의 모델/항 대응을 확인할 수 없습니다.", "모델 계수", "원본 행렬의 계수 스케일", "파일명과 모델 정의 확인 필요", "같은 RK 이름도 wafer 공통 회귀와 shot별 CPE 회귀를 구분해야 합니다.", False]
        grain = "shot별 CPE38" if model == "CPE" else "wafer WK/RK 동시 회귀"
        formula = terms[column]
        return [f"{grain}의 {coeff[3]} 방향 {coeff[1]}{coeff[2]} 계수 beta.", "CPE 계수" if model == "CPE" else "WK/RK 계수", "원본 행렬의 계수 스케일", f"term = ({formula}) × beta; x/y=fcp_x/y, rx/ry=local coordinate_X/Y", "계수와 point 제거값의 단위·크기를 혼동하지 마세요. 각 fit 값은 해당 term 기여의 합입니다.", True]
    penalty = re.fullmatch(r"(span|lambda)_(RK\d+_[XY])", column)
    if penalty and penalty[2] in CPE_TERMS:
        term = penalty[2]
        if penalty[1] == "span":
            return [f"shot CPE {term}의 virtual point별 term 기여 범위.", "CPE 진단", "측정값 원단위", f"max(eval_basis_{term} × beta) - min(eval_basis_{term} × beta)", "항별 span이며 전체 CPE_fit의 range가 아닙니다. RK1/2 상수 항은 span=0입니다.", True]
        return [f"shot CPE {term}에 실제 적용한 ridge penalty lambda.", "CPE 진단", "penalty 스케일", f"base_lambda × ridge_valve({term})", "RK1/2의 valve는 0입니다. lambda_max는 벡터 전체의 최대값입니다.", True]
    applied = re.fullmatch(r"(mrc_k|rk)(\d+)_valn", column)
    if applied and 1 <= int(applied[2]) <= 72:
        index = int(applied[2])
        axis = "X" if index % 2 else "Y"
        key = f"RK{index}_{axis}"
        kind = "PSM" if applied[1] == "mrc_k" else "TROCS"
        return [f"기적용 {kind} 72계수 중 {axis} 방향 K{index}.", f"기적용 {kind} 계수", "원본 행렬의 계수 스케일", f"term = ({APPLIED_TERMS[key]}) × coefficient; rx/ry=local 좌표", "재회귀한 wafer RK 또는 shot CPE 계수가 아닙니다. 홀수=X, 짝수=Y입니다.", True]
    cmp_coeff = re.fullmatch(r"(wk|rk)(\d+)", column)
    if cmp_coeff:
        index = int(cmp_coeff[2])
        axis = "X" if index % 2 else "Y"
        key = f"{cmp_coeff[1].upper()}{index}_{axis}"
        if key in WAFER_TERMS:
            return [f"PARAMDATA CMP 이력의 기적용 MRC {key} 계수.", "기적용 MRC 계수", "원본 행렬의 계수 스케일", f"MRC 평가 term = ({WAFER_TERMS[key]}) × coefficient; 전체 평가 후 -1 부호", "기적용 CMP 계수이며 이후 21번에서 raw로 다시 회귀한 WK/RK 계수와 다릅니다.", True]
    if column not in BASE_COLUMNS:
        return ["현재 제공한 노트북 기준 설명에 등록되지 않은 컬럼입니다.", "미등록", "확인 필요", "원본 DB/추가 가공 로직 확인 필요", "실제 컬럼을 누락하거나 의미를 추정하지 않고 목록에 남깁니다.", False]
    info = BASE_COLUMNS[column].copy()
    if column in ["pos_x_valn", "pos_y_valn"]:
        original_query = stage in ["12_PSM_SHOT_QUERY", "16_TROCS_QUERY"]
        processed = stage in ["12_PSM_INPUT", "16_TROCS_INPUT_UM"]
        if original_query:
            info[2], info[3] = "mm", "기적용 APC shot input DB 원본값"
        elif processed:
            info[2], info[3] = "µm", "DB 원본(mm) × 1000"
    if column == "apc_hist_index_no":
        if stage in ["12_PSM_SHOT_QUERY", "16_TROCS_QUERY"]:
            info[0] = "기적용 shot input 조회 결과의 EEXY/TROCS APC 이력 번호. 아직 main 번호로 변환하기 전입니다."
        else:
            info[0] = "노광/이력 연결 자료의 main APC 이력 번호. 가공 shot input은 EEXY/TROCS→main 연결 후 번호입니다."
    if column in ["raw_x", "raw_y"]:
        formulas = info[3].split("; ")
        if "ADI" in stage:
            info[3] = formulas[0]
        elif "OCO" in stage:
            info[3] = formulas[-1]
    if column == "status":
        if "CPE" in stage:
            info[3] = "ok / no_valid_points"
        elif "WK_RK" in stage:
            info[3] = "ok / skip_too_few_points"
    if column == "n_valid":
        if "CPE" in stage:
            info[3] = "CPE: raw_minus_total_fit_x/y + coordinate_X/Y가 모두 notna"
        elif "WK" in stage or "RK_WAFER" in stage:
            info[3] = "WK/RK: raw_x/y + fcp_x/y + coordinate_X/Y가 모두 notna"
    return info + [True]

def describe_file(stage):
    stage = normalize_stage(stage)
    mode = "ADI" if "ADI" in stage else "OCO" if "OCO" in stage else "공통"
    if stage.startswith(("02_", "03_")):
        return [mode, "LOT/wafer 이력", "조회 조건과 최신 이력을 통과한 LOTINFO 원본", "공통 wafer 선정에 사용할 LOT/slot/노광 이력과 공정 metadata"]
    if stage.startswith("04_"):
        return [mode, "LOT/wafer", "ADI/OCO 공통 wafer 선정 결과" if "PAIRED" not in stage else "ADI 기준 wafer에 OCO 이력을 붙인 paired metadata", "실측 point 데이터가 아니라 대상 wafer 목록입니다."]
    if stage.startswith(("05_", "06_")):
        return [mode, "data_kind별 원본 point/이력", "DB 원본 데이터 조회" if stage.startswith("05_") else "종류별 마지막 batch 선택 결과", "value/coordinate는 data_kind에 따라 의미가 달라집니다."]
    if stage.startswith("07_"):
        return [mode, "노광 이력/wafer", "EXPO 이력 최신 wafer 선정" if "EXPO" in stage else "F1 장비 metadata 조회", "측정값 보정 전 공정/장비 연결 자료입니다."]
    if stage.startswith("08_"):
        return [mode, "실측 point", "측정값·local/wafer 좌표·공정 정보를 결합한 point 자료", "MRC/PSM decorrect 전 값은 X_reg/Y_reg입니다."]
    if stage.startswith("09_"):
        return [mode, "CMP 이력/wafer", "최신 기적용 MRC WK1~20/RK3~20 계수", "소문자 wk/rk는 DB CMP 적용 계수입니다."]
    if stage.startswith("10_"):
        return [mode, "실측 point", "기적용 MRC 평가 및 decorrect", "X_reg_demrc/Y_reg_demrc까지 계산된 자료입니다."]
    if stage.startswith("11_"):
        return [mode, "APC 이력/LOT+sub_eqp", "main/EEXY 이력 조회 또는 최근접 이력 연결", "main 번호와 EEXY/TROCS 번호를 구분합니다."]
    if stage.startswith("12_"):
        return [mode, "APC/shot", "기적용 PSM 72계수 조회 및 main APC/좌표 연결", "QUERY pos=mm/EEXY 번호, INPUT pos=µm/main 번호입니다."]
    if stage.startswith("13_"):
        return [mode, "실측 point", "기적용 PSM 72계수 평가", "psm_fit은 계산값이며 ADI의 실제 PERSHOT 보정값은 PSM_X/Y입니다."]
    if stage.startswith("14_"):
        return [mode, "실측 point", "MRC/PSM decorrect 완료, 최종 장비 metadata 필터 전", "raw_x/y가 WK/RK 및 연구용 분석 입력입니다."]
    if stage.startswith("15_"):
        return [mode, "wafer" if "SUMMARY" in stage else "실측 point", "MRC/PSM decorrect 최종 결과" if "SUMMARY" not in stage else "decorrect 처리 상태의 wafer별 행 수 요약", "FINAL은 WK/RK와 CPE 제거 전입니다."]
    if stage.startswith("16_"):
        return [mode, "APC/shot", "기적용 TROCS RK1~72 조회 및 main APC 연결", "QUERY pos=mm/EEXY 번호, INPUT_UM pos=µm/main 번호입니다."]
    if "TROCS" in stage:
        if "ALLMARKS" in stage:
            return [mode, "UNIQUE_ID/shot/생성 mark", "wafer 전체 생성 mark의 TROCS 보정값", "실측 외의 생성 point도 포함합니다. TEST와 Test_ID를 구분하세요."]
        if "SHOT_COEFFICIENTS" in stage:
            return [mode, "UNIQUE_ID/shot", "TROCS 계산에 사용한 기적용 RK1~72 계수", "rk*_valn은 적용 이력이며 재회귀한 CPE RK가 아닙니다."]
        if "DIAGNOSTICS" in stage:
            return [mode, "UNIQUE_ID/shot 또는 미처리 wafer 사유", "TROCS 생성/연결 진단", "보정값 없는 후보 shot도 진단에 남깁니다."]
        if "ATTACHED" in stage:
            return [mode, "실측 point", "생성 TROCS 값의 실측 point 연결", "TROCS는 연결 값이며 raw에서 추가 차감하지 않습니다."]
        return [mode, "wafer" if "WAFER" in stage else "shot", "TROCS 보정값의 wafer/shot별 통계", "통계의 대상은 실측 residual이 아니라 기적용 TROCS 보정값입니다."]
    if stage.startswith("21_"):
        if "WAFER_COEFFICIENTS" in stage:
            return [mode, "wafer (LOT_ID+Wafer)", "WK/RK 동시 최소제곱 회귀의 WK 또는 RK 계수 분리", "동시 회귀 후 계수 block만 분리 저장했습니다."]
        if "DIAGNOSTICS" in stage:
            return [mode, "wafer", "WK/RK 회귀 상태·RMS·수치 rank 진단", "기본 유효점 19개 미만이면 skip, rank<19이면 해 비유일 가능성이 있습니다."]
        if "REMOVED_POINTS" in stage:
            return [mode, "실측 point", "point별 WK 또는 RK 제거값", "계수가 아니라 해당 좌표에서 실제 평가한 제거값입니다."]
        if stage.endswith("WK_RK_REMOVED"):
            return [mode, "실측 point", "WK/RK 제거 결과 및 shot CPE 회귀 입력", "CPE 입력은 raw_minus_total_fit_x/y입니다."]
        return [mode, "wafer" if "WAFER_DATA" in stage else "shot", "WK/RK 제거 전후와 제거 성분 통계", "RMS와 표본 표준편차는 다른 통계입니다."]
    if stage.startswith("27_"):
        if "COEFFICIENTS" in stage:
            return [mode, "shot (LOT_ID+Wafer+fcp_x/y)", "shot별 CPE38 RK 계수·span·lambda·회귀 진단", "RK*_X/Y는 해당 shot의 CPE38 재회귀 계수입니다."]
        if "DIAGNOSTICS" in stage:
            return [mode, "shot", "CPE38 회귀 및 G-opt/span 진단", "ok는 계산 완료 의미이며 cpe_quality_ok는 수치 진단입니다."]
        if "REMOVED_POINTS" in stage:
            return [mode, "실측 point", "각 point에서 제거한 CPE_fit_x/y", "계수 자체가 아니라 CPE matrix×shot 계수의 결과입니다."]
        if "FINAL_POINTS" in stage:
            return [mode, "실측 point", "원본 측정부터 최종 residual까지 포함한 최종 point 자료", "최종 분석값은 real_residual_x/y입니다."]
        return [mode, "wafer" if "WAFER_DATA" in stage else "shot", "CPE 입력·제거값·최종 residual 통계", "wafer 평균 RK 계수를 새로 계산한 파일이 아닙니다."]
    if stage.startswith("28_"):
        if "DECOMPOSITION" in stage:
            return [mode, "ADI/OCO × X/Y", "raw=WK+RK+CPE+최종 residual 분해식 검증", "검증 대상 0개이면 passed=False/check_status=no_valid_points입니다."]
        return [mode, "wafer" if "WAFER_DATA" in stage else "shot", "raw·TROCS 연결값·WK/RK·CPE·최종 residual의 전체 통계", "다음 연구 단계에서 전체 point가 필요하면 27_FINAL_POINTS를 사용하세요."]
    return [mode, "확인 필요", "현재 기준 노트북에 등록되지 않은 파일", "실제 파일명과 생성 코드를 확인하세요."]

def make_columns_table(stage, columns, file_label=None, dtypes=None):
    rows = []
    for order, column in enumerate(columns, start=1):
        explanation, category, unit, formula, note, known = describe_column(str(column), stage)
        rows.append([file_label or stage, stage, order, str(column), category, explanation, unit, formula, note, known, (dtypes or {}).get(str(column), "실제 파일 미검사")])
    return pd.DataFrame(rows, columns=SCHEMA_SHEET_COLUMNS)


## 03. 실제 stages 파일의 header/schema 검사
Parquet과 Excel이 함께 있으면 두 파일 모두 목록에 남깁니다. Excel sample 파일도 표시합니다. 읽기에 실패한 파일은 오류표에 남기고 나머지 파일을 계속 검사합니다.


In [ ]:
def inspect_header(path):
    if path.suffix.lower() == ".parquet":
        parquet = pq.ParquetFile(path)
        schema = parquet.schema_arrow
        return schema.names, {field.name: str(field.type) for field in schema}, parquet.metadata.num_rows
    workbook = load_workbook(path, read_only=True, data_only=False)
    try:
        sheet = workbook.worksheets[0]
        header = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True), ())
        columns = [str(value) if value is not None else f"[빈 헤더 {index}]" for index, value in enumerate(header, start=1)]
        return columns, {name: "Excel 헤더만 검사(자료형 미검사)" for name in columns}, max(0, (sheet.max_row or 1) - 1)
    finally:
        workbook.close()

files = sorted(path for path in STAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in [".parquet", ".xlsx"] and not path.name.startswith("~$"))
if not files:
    raise FileNotFoundError(f".parquet/.xlsx 파일이 없습니다: {STAGE_DIR}")
file_rows, column_tables, error_rows = [], [], []
for path in files:
    stage = normalize_stage(path.name)
    mode, grain, role, note = describe_file(stage)
    try:
        columns, dtypes, n_rows = inspect_header(path)
        column_tables.append(make_columns_table(stage, columns, path.name, dtypes))
        file_rows.append([path.name, stage, mode, grain, role, n_rows, len(columns), path.suffix[1:], bool(re.search(r"_sample_\d+$", path.stem)), note])
    except Exception as error:
        error_rows.append([path.name, type(error).__name__, str(error)])
df_files = pd.DataFrame(file_rows, columns=["파일명", "단계명", "ADI/OCO", "행 단위", "역할", "파일행수", "컬럼수", "형식", "Excel샘플여부", "해석·주의"])
df_columns = pd.concat(column_tables, ignore_index=True) if column_tables else pd.DataFrame(columns=SCHEMA_SHEET_COLUMNS)
df_errors = pd.DataFrame(error_rows, columns=["파일명", "오류종류", "오류내용"])
df_unknown = df_columns.loc[~df_columns["설명등록여부"].astype(bool)].copy()
display(df_files)
display(df_columns.head(30))
print(f"검사 성공 {len(df_files)}개 파일 / 실패 {len(df_errors)}개 / 컬럼 설명 {len(df_columns)}행 / 미등록 {len(df_unknown)}행")
if not df_errors.empty:
    display(df_errors)
if not df_unknown.empty:
    display(df_unknown)


## 04. 설명표를 Excel/CSV로 저장
Excel의 파일목록 / 파일별컬럼 / 미등록컬럼 / 읽기오류 / 참고 시트를 확인하세요. CSV는 UTF-8 BOM으로 저장하므로 한국어 Windows Excel에서 열 수 있습니다. 미등록 컬럼 설명을 01~02에서 보완한 뒤 03~04를 다시 실행하면 됩니다.


In [ ]:
def write_workbook(path, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name, index=False)
        for number, sheet in enumerate(writer.book.worksheets):
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            sheet.sheet_view.zoomScale = 85
            for cell in sheet[1]:
                cell.font = Font(name="맑은 고딕", bold=True, color="FFFFFF", size=10)
                cell.fill = PatternFill("solid", fgColor="193B50")
                cell.alignment = Alignment(vertical="center", wrap_text=True)
            sheet.row_dimensions[1].height = 28
            for row in sheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name="맑은 고딕", size=10)
                    cell.alignment = Alignment(vertical="top", wrap_text=True)
                    if isinstance(cell.value, str):
                        cell.data_type = "s"
                    if cell.row % 2 == 0:
                        cell.fill = PatternFill("solid", fgColor="F1F6F8")
                sheet.row_dimensions[row[0].row].height = 42
            for index, cell in enumerate(sheet[1], start=1):
                column = str(cell.value)
                width = 62 if column in ["설명", "출처·생성식", "해석·주의", "역할", "참고"] else 40 if column in ["파일명", "단계명", "전체Parquet위치", "예상 파일", "항 기여식"] else 24
                sheet.column_dimensions[get_column_letter(index)].width = width
            if sheet.max_row > 1:
                table = Table(displayName=f"GuideTable{number}", ref=sheet.dimensions)
                table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
                sheet.add_table(table)


GUIDE_NOTES = {
    '표 기준': '기존 MRC_PSM_decorrect.ipynb의 00~28 단계 SQL/생성식 기준입니다. Windows 실제 stages 파일은 여기서 읽지 않았습니다.',
    '검증 범위': 'DB 조회 필드 projection을 반영한 합성 실행에서 76개 저장 테이블 / 608개 고유 컬럼 / 파일별 컬럼 3157행을 확인했습니다. 실제 파일의 자료형·존재 여부·컬럼 수는 환경/결측 처리에 따라 달라질 수 있습니다.',
    '실제 파일 검사': 'Stages_Column_Dictionary.ipynb를 C:\\PHOTO_ANALYSIS\\BASIC에 놓고 실행하면 stages의 실제 Parquet schema/Excel 첫 행을 읽어 파일별 표를 새로 만듭니다.',
    '새 결과 폴더': 'C:\\PHOTO_ANALYSIS\\BASIC\\output_column_dictionary',
    '기본 입력': 'C:\\PHOTO_ANALYSIS\\BASIC\\output_decorrect_notebook\\stages',
    'Parquet와 Excel': '기본 SAVE_EXCEL=True이면 Excel은 stages에 있습니다. 조회 원본의 전체 Parquet은 save_snapshots=True일 때 snapshots에 있고, 가공 테이블은 stages에 저장됩니다.',
    'Excel 샘플': '_sample_10000.xlsx처럼 sample suffix가 붙은 Excel은 일부 행만 포함합니다. 헤더/컬럼 설명은 전체 컬럼이지만 통계 분석의 전체 입력으로 사용하면 행이 누락됩니다.',
    '대소문자': 'coordinate_x/y=원본 DB 필드; coordinate_X/Y=실측 local 좌표; Coordinate_X/Y=TROCS 생성 local 좌표입니다. 서로 다른 컬럼입니다.',
    '측정값 단위': '원본 노트북은 측정값 단위를 nm/µm로 추가 변환하지 않습니다. 측정값·보정값·residual은 입력 DB 측정값의 원단위를 유지하며 단위 확정은 사내 데이터 정의에 따릅니다.',
    '좌표 단위': '가공 좌표는 µm를 전제로 계산합니다. pos_x/y_valn의 조회 원본(mm)은 INPUT 단계에서 ×1000 변환합니다.',
    '계수 스케일': '계수는 해당 model matrix의 항/분모 스케일을 따릅니다. beta 자체를 point 보정값처럼 해석하지 말고 basis×beta를 확인하세요.',
    'RK 구분': 'rk3~20=DB CMP 기적용 MRC; rk1~72_valn=DB 기적용 TROCS; 21 RK*_X/Y=wafer 공통 재회귀; 27 RK*_X/Y=shot CPE38 재회귀입니다.',
    'raw 정의': 'X_reg/Y_reg는 최초 측정값, raw_x/y는 MRC·PSM decorrect 값입니다. ADI는 실제 PERSHOT, OCO는 계산한 PSM fit을 되돌립니다.',
    'TROCS 처리': 'TROCS 보정값은 계산해 연결합니다. 현재 raw→WK/RK→CPE 과정에서 TROCS를 추가 차감하지 않습니다.',
    '최종 residual': 'real_residual = raw - WK_fit - RK_fit - CPE_fit. WK/RK residual은 raw_minus_total_fit입니다.',
    '통계': '*_rms=sqrt(mean(value²)), *_std=표본 표준편차(ddof=1), *_mean=평균, *_max_abs=max(abs(value)), *_n_valid=개별 컬럼 non-NaN 수입니다.',
    'CPE 상태': 'status=ok는 회귀 계산 완료입니다. cpe_quality_ok는 G-opt/span 수치 조건이며 wafer 양불/rework 판정이 아닙니다.',
    'CPE inner/partial': '현재 코드에서는 n_valid와 n_virtual의 개수 비교로 판정합니다. Is_Edge_Shot의 네 모서리 기하 판정과 다릅니다.',
    '실제 파일 신규 컬럼': 'scanner는 등록되지 않은 컬럼도 미등록컬럼 시트에 남깁니다. 의미를 임의로 채우지 않습니다.',
}
df_notes = pd.DataFrame(GUIDE_NOTES.items(), columns=["항목", "참고"])
excel_path = OUTPUT_DIR / "stages_columns.xlsx"
write_workbook(excel_path, {"파일목록": df_files, "파일별컬럼": df_columns, "미등록컬럼": df_unknown, "읽기오류": df_errors, "참고": df_notes})
df_files.to_csv(OUTPUT_DIR / "stages_files.csv", index=False, encoding="utf-8-sig")
df_columns.to_csv(OUTPUT_DIR / "stages_columns.csv", index=False, encoding="utf-8-sig")
df_unknown.to_csv(OUTPUT_DIR / "stages_unknown_columns.csv", index=False, encoding="utf-8-sig")
for path in [excel_path, OUTPUT_DIR / "stages_files.csv", OUTPUT_DIR / "stages_columns.csv", OUTPUT_DIR / "stages_unknown_columns.csv"]:
    display(FileLink(path.relative_to(BASE_DIR).as_posix(), result_html_prefix="저장 파일: "))
print("컬럼 설명표 저장 완료:", excel_path)


## 05. 필요한 입력 파일 선택
MRC/PSM decorrect 결과는 `15_ADI_FINAL`/`15_OCO_FINAL`, WK/RK 제거 결과는 `21_ADI_WK_RK_REMOVED`/`21_OCO_WK_RK_REMOVED`, 최종 point 자료는 `27_ADI_FINAL_POINTS`/`27_OCO_FINAL_POINTS`입니다.

계수만 필요하면 `21_*_WK_WAFER_COEFFICIENTS`, `21_*_RK_WAFER_COEFFICIENTS`, `27_*_CPE_SHOT_COEFFICIENTS`를 구분해서 선택합니다. 전체 데이터 분석은 Parquet을 우선 사용하고 Excel sample의 행 수를 확인하세요.
